In [4]:
import torch

# torch 求导

参考 [url](https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)

pytorch 实现模型训练需要完整地写下训练过程，包括反向传播求梯度以及应用梯度下降算法。（06见chapter_2/03_...)

## 近似求导

In [5]:
def f(x):
    # 定义一个二次函数: f(x) = 3x^2 + 2x - 1
    # 其解析导数为 f'(x) = 6x + 2
    return 3. * x ** 2 + 2. * x - 1

# 近似求导函数
# 参数:
#   f: 需要求导的目标函数
#   x: 求导点
#   eps: 非常小的正数，用来近似计算极限 (默认1e-7)
def approximate_derivative(f, x, eps=1e-7):
    # 使用中心差分法近似 f 在 x 点的导数
    # [f(x+eps) - f(x-eps)] / (2*eps)
    # 这样做的原理是用 (x+eps) 和 (x-eps) 两个非常靠近 x 的点来构建割线，其斜率近似等于切线斜率
    return (f(x + eps) - f(x - eps)) / (2. * eps)

# 测试：在 x=1 处近似求函数 f 的导数
# 预期结果解析求导为: 6*1 + 2 = 8
print(approximate_derivative(f, 1.))

8.000000000230045


In [6]:
# 求函数 g(x1, x2) 对各自变量的偏导数
# g(x1, x2) 是一个关于两个变量的函数，其中一个变量不变，对另一个变量求导
def g(x1, x2):
    # 定义一个简单的函数：g(x1, x2) = (x1 + 5) * (x2 ** 2)
    # 对 x1 求偏导时，x2 视为常数
    # 对 x2 求偏导时，x1 视为常数
    return (x1 + 5) * (x2 ** 2)

# 实现求偏导数的关键思想其实就是：  
# 我们把一个多变量函数（比如 g(x1, x2)）当成关于一个变量的单变量函数，  
# 固定其他变量不变，仅对其中一个变量做微小变化，然后用“中心差分法”计算该点的斜率。
# approximate_derivative(lambda x: g(x, x2), x1) 这句代码意思是：  
# 固定 x2，只让 x1 变动，于是 g 就成了关于 x1 的单变量函数。  
# approximate_derivative 会返回 g 对 x1 的偏导数的近似值。
# 同理 approximate_derivative(lambda x: g(x1, x), x2) 是固定 x1，只对 x2 微调，  
# 算出来的就是 g 对 x2 的偏导（即偏导向量的第二个分量）。

def approximate_gradient(g, x1, x2, eps=1e-3):
    # 近似计算 g 在点 (x1, x2) 处的梯度（偏导数）
    # 使用中心差分法分别对 x1 和 x2 求偏导
    # dg_x1: 偏 g / 偏 x1，固定 x2，只对 x1 进行微小扰动
    dg_x1 = approximate_derivative(lambda x: g(x, x2), x1, eps)
    # dg_x2: 偏 g / 偏 x2，固定 x1，只对 x2 进行微小扰动
    dg_x2 = approximate_derivative(lambda x: g(x1, x), x2, eps)
    # 返回在 (x1, x2) 处对 x1 和 x2 的偏导数组成的梯度向量
    return dg_x1, dg_x2

# 测试：在 x1=2, x2=3 的点上，计算 g(x1, x2) 的偏导
# 预期：实际的解析偏导为
# 偏 g / 偏 x1 = 1 * (x2 ** 2) = 9.0
# 偏 g / 偏 x2 = 2 * (x1 + 5) * x2 = 2*7*3=42.0
print(approximate_gradient(g, 2., 3.))

(8.999999999993236, 41.999999999994486)


## torch 近似求导

In [7]:
# 声明两个 tensor 变量 x1 和 x2，类型为 float，分别赋值为 2 和 3，并设置 requires_grad=True 表示这两个变量需要计算梯度（后续可以进行自动求导）
x1 = torch.tensor([2.], requires_grad=True)    # x1 作为自变量，可计算梯度
x2 = torch.tensor([3.], requires_grad=True)    # x2 作为自变量，可计算梯度

# 计算函数 g(x1, x2) 的值，g 已经前面实现，是关于两个变量的函数，这里以 tensor 作为参数
y = g(x1, x2)    # 此时 y 也是一个可以自动求导的 tensor，实际就是 (2+5)*(3**2)=7*9=63

# 使用 torch.autograd.grad 计算 y 对 x1 的偏导数
# - y: 要求导的目标 tensor
# - x1: 需要对谁求导（可以是 tensor 或列表，这里想得到 dy/dx1）
x1 = torch.tensor([2.], requires_grad=True)
x2 = torch.tensor([3.], requires_grad=True)
y = g(x1, x2)
    
(dy_dx1,) = torch.autograd.grad(y, x1,retain_graph=True)
print(dy_dx1)


tensor([9.])


In [ ]:
# 这里我们尝试计算 y 对 x2 的偏导数。注意：如果 retain_graph=False（默认），
# 计算后自动释放构建的计算图，所以后续不能再次对同一结果反向传播或再次求导。
try:
    # 这里由于上面对 x1 的求导已经 retain_graph=True，
    # 但此处我们使用 retain_graph=False（或省略），表示只保留一次图。
    # 若前面已经释放过了，这里会报错“Trying to backward through the graph a second time.”
    (dy_dx2,) = torch.autograd.grad(y, x2, retain_graph=False)
    print(dy_dx2)
except Exception as e:
    # 如果由于前面已经使用过计算图，这里捕获并打印异常信息
    print(e)

Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.


In [11]:
# 同时求导

x1 = torch.tensor([2.], requires_grad=True)
x2 = torch.tensor([3.], requires_grad=True)
y = g(x1, x2)

# 求偏导数
dy_dx1, dy_dx2 = torch.autograd.grad(y, [x1, x2])


print(dy_dx1, dy_dx2)

tensor([9.]) tensor([42.])


In [12]:
# 当然我们一般直接用 backward

x1 = torch.tensor([2.], requires_grad=True)
x2 = torch.tensor([3.], requires_grad=True)
y = g(x1, x2)

# 求偏导数,求梯度
y.backward()
print(x1.grad, x2.grad)  # 优化器 x1-学习率*x1.grad

tensor([9.]) tensor([42.])


## 二阶导


In [11]:
# 定义可求导变量 x1 和 x2，分别赋值为 2. 和 3.
x1 = torch.tensor([2.], requires_grad=True)
x2 = torch.tensor([3.], requires_grad=True)
# 计算函数 g 在 (x1, x2) 处的值，得到标量 y
y = g(x1, x2)

# 计算 y 对 x1 和 x2 的一阶偏导 dy/dx1, dy/dx2
# create_graph=True 让计算得到的梯度也可以继续用于二阶导的求取
dy_dx1, dy_dx2 = torch.autograd.grad(y, [x1, x2], create_graph=True)

# 计算 dy/dx1 对 x1 和 x2 的二阶偏导：即 d²y/dx1², d²y/dx1dx2
# allow_unused=True 允许返回的梯度为 None（如没有依赖关系的情况），以免出错
dy_dx1_dx1, dy_dx1_dx2 = torch.autograd.grad(dy_dx1, [x1, x2], allow_unused=True)
# 计算 dy/dx2 对 x1 和 x2 的二阶偏导：即 d²y/dx2dx1, d²y/dx2²
dy_dx2_dx1, dy_dx2_dx2 = torch.autograd.grad(dy_dx2, [x1, x2], allow_unused=True)

# 打印所有二阶偏导：d²y/dx1², d²y/dx2dx1, d²y/dx2²
print(dy_dx1_dx1, dy_dx2_dx1, dy_dx2_dx2)

None tensor([6.]) tensor([14.])


# 手动模拟实现优化器

In [12]:
# 模拟梯度下降算法（SGD － 随机梯度下降）
import torch

# 设置学习率
learning_rate = 0.3

# 初始化参数 x，这里用 2.0 作为初值，并声明需要计算梯度
x = torch.tensor(2.0, requires_grad=True)

# 用于迭代优化 100 次
for step in range(100):
    # 计算目标函数 f(x) 的值，设为 z
    z = f(x)
    
    # 从第二步开始，每一步都需要将上一步计算得到的梯度清零，否则梯度会累加
    if step > 0:
        x.grad.zero_() # 等价于 x.grad = 0，防止梯度累积
    
    # 自动求导，计算 z 对 x 的梯度，结果存储在 x.grad
    z.backward()
    
    # 用梯度下降法更新参数 x
    # x.data.sub_(learning_rate * x.grad) 等价于 x = x - learning_rate * x.grad
    # 这里 .data 是直接对数据进行原地修改，避免影响计算图
    x.data.sub_(learning_rate * x.grad)

# 输出优化后的参数 x
print(x)

tensor(-0.3333, requires_grad=True)


In [13]:
a=torch.tensor(2) # 标量
a.shape

torch.Size([])

In [ ]:
# GradientTape与optimizer（优化器）结合使用，下面演示如何用PyTorch的优化器来优化变量x
learning_rate = 0.3  # 设置学习率为0.3
x = torch.tensor(2.0, requires_grad=True)  # 初始化参数x为2.0，并声明需要计算梯度
# 创建一个SGD优化器，指定待优化参数x，学习率lr=0.3，以及momentum=0.9
optimizer = torch.optim.SGD([x], lr=learning_rate, momentum=0.9)

# 进行220轮参数更新
for _ in range(220):
    z = f(x)  # 计算目标函数f(x)，得到当前的损失/目标值z
    optimizer.zero_grad()  # 梯度清零，避免梯度累积（上一轮的梯度需要清零）
    z.backward()  # 自动求导，计算z对x的梯度，结果保存在x.grad中
    # print(x.grad)  # 可选：输出梯度，便于调试和观察
    optimizer.step()  # 执行一步参数更新：x = x - learning_rate * x.grad，同时考虑动量参数
    
# 输出经过优化后的参数x的最终值
print(x)


tensor(-0.3333, requires_grad=True)
